In [ ]:
import os
import re
from pathlib import Path
from textwrap import dedent
from typing import Dict, List, Optional, Self, Sequence, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import spearmanr

from utils import kernel_quantile_band_rankx

hf_path = (
    "/home/stenheli/projects/heart-failure-detection/sandbox/ensembled_testset.csv"
)
df = pd.read_csv(hf_path)
echo_path = "/dataset/ecg/tabular/raw/echo_22092025.csv"
echo = pd.read_csv(echo_path)
fnr_map_path = "/dataset/ecg/tabular/raw/2025_32_echo_mapping_foedselsnr.csv"
fnr_map = pd.read_csv(fnr_map_path)

echo = echo.merge(
    fnr_map[["FOEDSELSNR_SINGLE", "FOEDSELSNR_DOUBLE"]],
    left_on="ID",
    right_on="FOEDSELSNR_SINGLE",
    how="left",
)
echo = echo[~echo["FOEDSELSNR_DOUBLE"].isna()]
echo = echo.rename(columns={"FOEDSELSNR_DOUBLE": "FOEDSELSNR"})
echo = echo.drop(columns=["FOEDSELSNR_SINGLE"])

MAX_DAYS = 3
N_GRID = 400
QS = (0.25, 0.75)

CACHE_DIR = "./cache/"
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

# Define mapping from parameters to field names from the echo database

In [ ]:
CONFIG_PATH = Path("echo_settings.yaml")

with CONFIG_PATH.open("r") as f:
    cfg = yaml.safe_load(f)

COMPACS_MAPPING = cfg["COMPACS_MAPPING"]
UNITS_MAPPING = cfg["UNITS_MAPPING"]
COMORBIDITY_MAP = cfg["COMORBIDITY_MAP"]
PARAMETERS = cfg["PARAMETERS"]

echo_min_max_cutoff_mapping = {
    k: tuple(v) for k, v in cfg["MIN_MAX_CUTOFF_MAPPING"].items()
}

COMORBIDITY_REGEX = cfg["COMORBIDITY_REGEX"]
PLOT_LIMS = cfg["PLOT_LIMS"]

N_BOOTSTRAP = cfg["N_BOOTSTRAP"]

# Keep patients with both echo and ECG

In [ ]:
df_pids = df["FOEDSELSNR"]
echo_pids = echo["FOEDSELSNR"]
overlap = np.intersect1d(df_pids, echo_pids)
df = df[df["FOEDSELSNR"].isin(overlap)]
echo = echo[echo["FOEDSELSNR"].isin(overlap)]

# Backwards fill dataframe; the raw version contains one row per edit

In [ ]:
def first_available_column(df: pd.DataFrame, candidates):
    """Return the first column from candidates that exists in df, else None."""
    for c in candidates:
        if c in df.columns:
            return c
    return None


def coalesce_series(df: pd.DataFrame, candidates):
    """
    Return a single Series coalescing across candidates (first non-null per row),
    using only columns present in df. Returns (series, cols_used).
    """
    avail = [c for c in candidates if c in df.columns]
    if not avail:
        return pd.Series(index=df.index, dtype=float), []
    s = df[avail].bfill(axis=1).iloc[:, 0]
    return s, avail


group_series = {}
chosen_columns = {}
for grp, candidates in COMPACS_MAPPING.items():
    s, used = coalesce_series(echo, candidates)
    group_series[grp] = s
    chosen_columns[grp] = used

meta_cols = ["SeriesDate", "FOEDSELSNR"]
for col in meta_cols:
    if col in echo.columns:
        group_series[col] = echo[col]

echo_hf_grouped = pd.DataFrame(group_series, index=echo.index)

In [ ]:
# Drop outliers in E and A, then compute E/A
E_lower, E_upper = echo_min_max_cutoff_mapping["E"]
A_lower, A_upper = echo_min_max_cutoff_mapping["A"]

E = echo_hf_grouped["E"].copy()
A = echo_hf_grouped["A"].copy()

E[(E < E_lower) | (E > E_upper)] = np.nan
A[(A < A_lower) | (A > A_upper)] = np.nan

echo_hf_grouped["E"] = E
echo_hf_grouped["A"] = A
echo_hf_grouped["E/A"] = E / A

echo_hf_grouped["BSA"] = echo_hf_grouped["CO"] / echo_hf_grouped["CI"]
echo_hf_grouped["LVEDVi"] = echo_hf_grouped["LVEDVOL"] / echo_hf_grouped["BSA"]

LAv_lower, LAv_upper = echo_min_max_cutoff_mapping["LAv"]
echo_hf_grouped["LAv"] = echo_hf_grouped["LAv"].where(
    (echo_hf_grouped["LAv"] >= LAv_lower) & (echo_hf_grouped["LAv"] <= LAv_upper)
)
echo_hf_grouped["LAVi"] = echo_hf_grouped["LAv"] / echo_hf_grouped["BSA"]
echo_hf_grouped["LVMi"] = echo_hf_grouped["LVM"] / echo_hf_grouped["BSA"]

echo_hf_grouped["e'"] = echo_hf_grouped["e'"] * 100  # m/s to cm/s

# Drop redundant/non-indexed volumes
echo_hf_grouped = echo_hf_grouped.drop(columns=["LAv", "LVEDVOL", "CO"])

# Filter outliers in echo dataframe

In [ ]:
echo_hf_grouped = echo_hf_grouped.copy()

for name in echo_min_max_cutoff_mapping.keys():
    if name not in echo_hf_grouped.columns:
        continue

    count_before = echo_hf_grouped[name].notna().sum()

    lower, upper = echo_min_max_cutoff_mapping.get(name, (None, None))
    echo_hf_grouped.loc[echo_hf_grouped[name] < lower, name] = np.nan
    echo_hf_grouped.loc[echo_hf_grouped[name] > upper, name] = np.nan

    count_after = echo_hf_grouped[name].notna().sum()

    print(f"{name:<12}: {count_before:,} -> {count_after:,}")

In [ ]:
echo_hf_grouped["SeriesDate"] = pd.to_datetime(
    echo_hf_grouped["SeriesDate"], errors="coerce"
)
df["time"] = pd.to_datetime(df["time"], errors="coerce")

echo_hf_grouped = echo_hf_grouped.sort_values(by=["FOEDSELSNR", "SeriesDate"])
echo_hf_grouped.update(echo_hf_grouped.groupby("FOEDSELSNR").bfill())


print(len(echo_hf_grouped))
echo_first = (
    echo_hf_grouped.dropna(subset=["FOEDSELSNR", "SeriesDate"])
    .sort_values(["FOEDSELSNR", "SeriesDate"])
    .drop_duplicates(subset=["FOEDSELSNR"], keep="first")
    .reset_index(drop=True)
)
print(len(echo_first))


df_times = df.dropna(subset=["FOEDSELSNR", "time"])[["FOEDSELSNR", "time"]].copy()

cand = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times, on="FOEDSELSNR", how="left"
)

cand["abs_diff"] = (cand["time"] - cand["SeriesDate"]).abs()

idx = (
    cand.dropna(subset=["time"])  # only patients with at least one ECG
    .groupby("FOEDSELSNR")["abs_diff"]
    .idxmin()
)

nearest = cand.loc[idx].copy()

no_ecg_patients = set(echo_first["FOEDSELSNR"]) - set(nearest["FOEDSELSNR"])
if no_ecg_patients:
    nearest = pd.concat(
        [
            nearest,
            echo_first.loc[
                echo_first["FOEDSELSNR"].isin(no_ecg_patients),
                ["FOEDSELSNR", "SeriesDate"],
            ].assign(time=pd.NaT, abs_diff=pd.NaT),
        ],
        ignore_index=True,
    )

matched = nearest.merge(
    echo_first, on=["FOEDSELSNR", "SeriesDate"], how="left", suffixes=("", "_echo")
)

matched["delta"] = matched["time"] - matched["SeriesDate"]
matched["delta_days"] = matched["delta"].dt.total_seconds() / (24 * 3600)
matched["abs_delta_days"] = matched["delta_days"].abs()

print(f"First-echo patients: {len(echo_first)}")
print(f"Matched with at least one ECG: {matched['time'].notna().sum()}")
print(f"No ECG found: {matched['time'].isna().sum()}")

In [ ]:
td_limit = pd.Timedelta(days=MAX_DAYS)

df_times = df.dropna(subset=["FOEDSELSNR", "time"])[
    ["FOEDSELSNR", "time", "ensemble", "age", "sex", "hf_icd_flag_ever"]
].copy()

cand = echo_first[["FOEDSELSNR", "SeriesDate"]].merge(
    df_times, on="FOEDSELSNR", how="left"
)
cand["abs_diff"] = (cand["time"] - cand["SeriesDate"]).abs()

idx = cand.dropna(subset=["time"]).groupby("FOEDSELSNR")["abs_diff"].idxmin()

nearest = cand.loc[idx].copy()

no_ecg_patients = set(echo_first["FOEDSELSNR"]) - set(nearest["FOEDSELSNR"])
if no_ecg_patients:
    nearest = pd.concat(
        [
            nearest,
            echo_first.loc[
                echo_first["FOEDSELSNR"].isin(no_ecg_patients),
                ["FOEDSELSNR", "SeriesDate"],
            ].assign(
                time=pd.NaT, abs_diff=pd.NaT, ensemble=np.nan, age=np.nan, sex=np.nan
            ),
        ],
        ignore_index=True,
    )

matched = nearest.merge(
    echo_first, on=["FOEDSELSNR", "SeriesDate"], how="left", suffixes=("", "_echo")
)

matched["delta"] = matched["time"] - matched["SeriesDate"]
matched["delta_days"] = matched["delta"].dt.total_seconds() / (24 * 3600)
matched["abs_delta_days"] = matched["delta_days"].abs()

keep_mask = matched["delta"].notna() & (matched["delta"].abs() <= td_limit)
kept = matched.loc[keep_mask].copy()
kept["age"] = kept["age"].str.extract(r"(\d+)").astype(int)

# drop age < 18 and age > 100 and print then umber of dropped patients
age_mask = (kept["age"] >= 18) & (kept["age"] <= 100)
dropped_age = len(kept) - age_mask.sum()
if dropped_age > 0:
    print(f"Dropping {dropped_age} patients due to age < 18 or > 100")
kept = kept.loc[age_mask]
sex_map = {"M": 1.0, "F": 0.0, "O": 0.5}
kept["sex"] = kept["sex"].map(sex_map).astype(float)

In [ ]:
print(f"Total first-echo patients: {len(matched)}")
print(f"Kept within ±{MAX_DAYS} days: {len(kept)}")
print(f"Dropped: {len(matched) - len(kept)}")

s = kept["delta_days"]
if not s.empty:
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.hist(-s, bins=int(MAX_DAYS * 24 + 1), edgecolor="C0")
    ax.set_xlabel(f"Time from ECG to echocardiography (days)")
    ax.set_ylabel("Patients")
    fig.tight_layout()
    ax.set_xticks(np.arange(-int(MAX_DAYS), int(MAX_DAYS) + 1, 1))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_yticks([1000, 2000, 3000], labels=["1k", "2k", "3k"])
    plt.savefig("figures/png/ecg_echo_time_difference_histogram.png", dpi=300)
    plt.savefig(
        "figures/pgf/ecg_echo_time_difference_histogram.pgf", bbox_inches="tight"
    )
    plt.show()

    mean_abs_diff = s.abs().mean()
    print(
        f"Mean absolute time difference between ECG and Echo: {mean_abs_diff*24:.2f} hours"
    )
    # count how often before vs affter
    before_count = (s < 0).sum()
    after_count = (s > 0).sum()
    print(f"ECG before Echo: {before_count} times")
    print(f"ECG after Echo: {after_count} times")

else:
    print("No pairs within the specified time window.")

In [ ]:
cache_file_name = "diagnoses.csv"
if not os.path.exists(os.path.join(CACHE_DIR, cache_file_name)):
    diag = pd.read_csv("/dataset/ecg/tabular/raw/2025_32_Diagnoser.csv", dtype="str")
    demo = pd.read_csv("/dataset/ecg/tabular/raw/2025_32_Demorgafi.csv", dtype="str")
    demo_pids = demo["PERSONID"].unique().tolist()
    diag_pids = diag["PERSONID"].unique().tolist()
    common_pids = set(demo_pids).intersection(set(diag_pids))
    demo_foedselsnrs = demo["FOEDSELSNR"].unique().tolist()
    echo_foedselsnrs = kept["FOEDSELSNR"].unique().tolist()
    common_foedselsnrs = set(demo_foedselsnrs).intersection(set(echo_foedselsnrs))
    subdemo = demo[demo["FOEDSELSNR"].isin(common_foedselsnrs)].copy()
    subdiag = diag[diag["PERSONID"].isin(subdemo["PERSONID"].unique())].copy()
    # add the FOEDSELSNR to subdiag by merging on PERSONID
    subdiag = subdiag.merge(
        subdemo[["PERSONID", "FOEDSELSNR"]], on="PERSONID", how="left"
    )

    # endow subdiag with ecg_date, taken from the "kept" dataframe
    subdiag = subdiag.merge(
        kept[["FOEDSELSNR", "time"]].drop_duplicates(), on="FOEDSELSNR", how="left"
    )
    subdiag.to_csv(os.path.join(CACHE_DIR, cache_file_name), index=False)
else:
    subdiag = pd.read_csv(os.path.join(CACHE_DIR, cache_file_name), dtype="str")
subdiag["DIAGNOSETID"] = pd.to_datetime(subdiag["DIAGNOSETID"])
subdiag["time"] = pd.to_datetime(subdiag["time"])

# from "df", add the age and sex columns to subdiag by merging on PERSONID
df["age"] = pd.to_numeric(df["age"].str.replace("Y", ""))
df["sex"] = df["sex"].str.lower()
subdiag = subdiag.merge(df[["PERSONID", "age", "sex"]], on="PERSONID", how="left")

subdiag["IS_BASELINE"] = (
    subdiag["DIAGNOSETID"] <= subdiag["time"] + pd.Timedelta(days=-1)
).astype(int)

In [ ]:
for comorbidity, patterns in COMORBIDITY_REGEX.items():
    kept[comorbidity] = 0
    mask = subdiag["KODE"].str.match("|".join(patterns))
    affected_foedselsnrs = subdiag.loc[mask, "FOEDSELSNR"].unique().tolist()
    kept.loc[kept["FOEDSELSNR"].isin(affected_foedselsnrs), comorbidity] = 1

    print(f"Comorbidity '{comorbidity}': {kept[comorbidity].sum()} patients")

In [ ]:
plot_cols = [c for c in PARAMETERS if c not in ("sex", "age")]

kept_filtered = kept.copy()

rng = np.random.default_rng(0)  # for reproducibility

corr_results = []
for col in [c for c in plot_cols if c != "sex"]:
    if col == "FOEDSELSNR":
        continue

    x = kept_filtered[col].to_numpy()
    y = kept_filtered["ensemble"].to_numpy()

    # drop NaNs for the pair
    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = x.size  # <- n after pairwise NaN filtering

    if n < 3:
        # not enough data to compute correlation reliably
        continue

    # point estimate
    corr, _ = spearmanr(x, y)

    # bootstrap CI
    boot_corrs = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.integers(0, n, size=n)
        bx = x[idx]
        by = y[idx]
        bcorr, _ = spearmanr(bx, by)
        if np.isfinite(bcorr):
            boot_corrs.append(bcorr)

    boot_corrs = np.asarray(boot_corrs)
    ci_low = np.percentile(boot_corrs, 2.5)
    ci_high = np.percentile(boot_corrs, 97.5)

    corr_results.append((col, n, corr, ci_low, ci_high))

corr_df = pd.DataFrame(
    corr_results,
    columns=["parameter", "n", "spearman_corr", "ci_low", "ci_high"],
)

# sort plot_cols by absolute correlation (using point estimate)
plot_cols = sorted(
    plot_cols,
    key=lambda c: (
        abs(corr_df.loc[corr_df["parameter"] == c, "spearman_corr"].values[0])
        if (corr_df["parameter"] == c).any()
        else -np.inf
    ),
    reverse=True,
)

corr_df = (
    corr_df.assign(abs_corr=lambda d: d["spearman_corr"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

rows = []
for _, row in corr_df.iterrows():
    rows.append(
        "      "
        f"{row['parameter']} & "
        f"\\num{{{int(row['n'])}}} & "
        f"\\num{{{row['spearman_corr']:.2f}}} & "
        f"\\num{{{row['ci_low']:.2f}}} & "
        f"\\num{{{row['ci_high']:.2f}}} \\\\"
    )
body = "\n".join(rows) + "\n"

tabular = (
    r"   {\setlength{\tabcolsep}{12pt}"
    "\n   \\begin{tabular}{lcrrr}\n"
    "      \\toprule\n"
    "      Parameter & $n$ & $Spearman \\rho$ & "
    " \\multicolumn{2}{c}{95\\% CI} \\\\\n"
    "      & & & Low & High \\\\\n"
    "      \\midrule\n"
    f"{body}"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
    "   }\n"
)

latex = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and ensemble prediction.}\n"
    "   \\label{tab:spearman_corr}\n"
    f"{tabular.replace('_', ' ')}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item $n$ is the number of non-missing (parameter, ensemble) pairs used for each correlation. "
    "Higher absolute $\\rho$ indicates stronger monotonic association. "
    "Values are 95\\% bootstrap confidence intervals (\\num{10000} resamples).\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

with open("tables/spearman_table.tex", "w") as f:
    f.write(latex)

In [ ]:
def compute_corr_for_df(df, param_cols, target_col="ensemble"):
    out = []
    for col in param_cols:
        if col in ("sex", "FOEDSELSNR"):
            continue
        corr, _ = spearmanr(df[col], df[target_col], nan_policy="omit")
        out.append((col, corr))
    return pd.DataFrame(out, columns=["parameter", "rho"])


overall_corr_df = compute_corr_for_df(kept_filtered, plot_cols)

# sort by |corr| overall
overall_corr_df = (
    overall_corr_df.assign(abs_corr=lambda d: d["rho"].abs())
    .sort_values("abs_corr", ascending=False)
    .drop(columns=["abs_corr"])
)

param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

lvef_groups = {
    r"$\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"41--49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"$\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}

lvef_corrs = {}
for label, mask in lvef_groups.items():
    df_sub = kept_filtered.loc[mask]
    # compute on the same param_order
    rhos = []
    for col in param_order:
        corr, _ = spearmanr(df_sub[col], df_sub["ensemble"], nan_policy="omit")
        rhos.append(corr)
    lvef_corrs[label] = rhos

sex_values = [v for v in kept_filtered["sex"].dropna().unique()]

sex_label_map = {val: f"sex = {val}" for val in sex_values}

sex_corrs = {}
for val in sex_values:
    label = sex_label_map[val]
    df_sub = kept_filtered.loc[kept_filtered["sex"] == val]
    rhos = []
    for col in param_order:
        corr, _ = spearmanr(df_sub[col], df_sub["ensemble"], nan_policy="omit")
        rhos.append(corr)
    sex_corrs[label] = rhos


def perm_test_diff_spearman_overall(df, param, target="ensemble", B=1000, seed=123):
    """
    Two-sided permutation test p-value for difference in Spearman correlations
    between males (sex == 1.0) and females (sex == 0.0), for a given parameter.
    """
    rng = np.random.default_rng(seed)

    sub = df[[param, target, "sex"]].dropna()

    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    # observed difference
    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        # if a permutation accidentally gives too few of one sex, skip by setting diff to 0
        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p


# Compute sample sizes for sex groups
n_male = kept_filtered.loc[kept_filtered["sex"] == 1.0].shape[0]
n_female = kept_filtered.loc[kept_filtered["sex"] == 0.0].shape[0]

# Existing correlations
sex_cols = list(sex_corrs.keys())
n_sex = len(sex_cols)
lvef_cols = list(lvef_groups.keys())
n_lvef = len(lvef_cols)

# ===== NEW: first collect all rhos to find global max |rho| =====
row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    # LVEF correlations for this parameter
    rhos_lvef = [
        spearmanr(
            kept_filtered.loc[lvef_groups[c], param],
            kept_filtered.loc[lvef_groups[c], "ensemble"],
            nan_policy="omit",
        )[0]
        for c in lvef_cols
    ]
    # Sex correlations
    rho_male = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 1.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 1.0, "ensemble"],
        nan_policy="omit",
    )[0]
    rho_female = spearmanr(
        kept_filtered.loc[kept_filtered["sex"] == 0.0, param],
        kept_filtered.loc[kept_filtered["sex"] == 0.0, "ensemble"],
        nan_policy="omit",
    )[0]

    # permutation p-value (male vs female)
    pval = perm_test_diff_spearman_overall(kept_filtered, param, B=N_BOOTSTRAP)

    # store row-wise
    row_data.append(
        {
            "pname": pname,
            "rhos_lvef": rhos_lvef,
            "rho_male": rho_male,
            "rho_female": rho_female,
            "pval": pval,
        }
    )

    # collect absolute rhos (skip NaNs)
    for r in rhos_lvef + [rho_male, rho_female]:
        if not np.isnan(r):
            all_abs_rhos.append(abs(r))

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0


def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    # wrap the number in \num
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"


rows = []
for rd in row_data:
    lvef_cells = [fmt_rho_colored(r, max_abs_rho) for r in rd["rhos_lvef"]]
    male_cell = fmt_rho_colored(rd["rho_male"], max_abs_rho)
    female_cell = fmt_rho_colored(rd["rho_female"], max_abs_rho)
    pval = rd["pval"]

    if pd.isna(pval):
        p_str = ""
    else:
        if pval < 0.001:
            # cap at <0.001, numeric part wrapped in \num
            p_str = r"<\num{0.001}"
        else:
            p_str = rf"\num{{{pval:.3f}}}"

    row = (
        "      "
        + " & ".join([rd["pname"]] + lvef_cells + [male_cell, female_cell, p_str])
        + r" \\"
    )
    rows.append(row)

col_spec_all = "l" + "r" * (n_lvef + n_sex + 1)

header_top = (
    "      & "
    + f"\\multicolumn{{{n_lvef}}}{{c}}{{LVEF subgroup}} & "
    + f"\\multicolumn{{{n_sex + 1}}}{{c}}{{Sex}} \\\\"
)

header_mid = "       & " + " & ".join(lvef_cols + ["Male", "Female", "p"]) + r" \\"

cmidrule_lvef = f"      \\cmidrule(lr){{2-{1 + n_lvef}}}"
cmidrule_sex = f"      \\cmidrule(lr){{{2 + n_lvef}-{1 + n_lvef + n_sex + 1}}}"

combined_tabular = (
    r"   {\setlength{\tabcolsep}{4pt}"
    f"   \\begin{{tabular}}{{{col_spec_all}}}\n"
    "\n      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrule_lvef}\n"
    f"{cmidrule_sex}\n"
    f"{header_mid}\n"
    "      \\midrule\n" + "\n".join(rows) + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
    "    }\n"
)

combined_tabular = combined_tabular.replace("_", " ")

latex_combined = (
    "\\begin{table}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and ensemble prediction by LVEF subgroup and sex.}\n"
    "   \\label{tab:spearman_corr_lvef_sex}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "The $p$ column reports two-sided permutation test $p$-values for the difference in correlation between males and females, "
    "based on \\num{10000} random reassignments of sex labels. "
    "Values are capped at $<\\num{0.001}$.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table}\n"
)

with open("tables/spearman_table_lvef_sex_p.tex", "w") as f:
    f.write(latex_combined)

In [ ]:
param_order = overall_corr_df["parameter"].tolist()
pretty_names = overall_corr_df["parameter"].tolist()

# ---------- LVEF subgroups ----------
lvef_groups = {
    r"LVEF $\leq$ 40\%": kept_filtered["LVEF"] <= 40,
    r"LVEF 41--49\%": (kept_filtered["LVEF"] > 40) & (kept_filtered["LVEF"] < 50),
    r"LVEF $\geq$ 50\%": kept_filtered["LVEF"] >= 50,
}
lvef_cols = list(lvef_groups.keys())


def perm_test_diff_spearman(df, l_mask, param, target="ensemble", B=1000, seed=123):
    rng = np.random.default_rng(seed)

    # subset to this LVEF group and needed columns
    sub = df.loc[l_mask, [param, target, "sex"]].dropna()

    # split by sex
    m = sub[sub["sex"] == 1.0]
    f = sub[sub["sex"] == 0.0]

    if len(m) < 4 or len(f) < 4:
        return np.nan

    # observed difference
    rho_m, _ = spearmanr(m[param], m[target])
    rho_f, _ = spearmanr(f[param], f[target])
    obs_diff = rho_m - rho_f

    diffs = np.empty(B)

    for b in range(B):
        # permute sex labels
        permuted = sub.copy()
        permuted["sex"] = rng.permutation(permuted["sex"].values)

        m_p = permuted[permuted["sex"] == 1.0]
        f_p = permuted[permuted["sex"] == 0.0]

        # if a permutation accidentally gives too few of one sex, skip
        if len(m_p) < 4 or len(f_p) < 4:
            diffs[b] = 0.0
            continue

        rho_m_p, _ = spearmanr(m_p[param], m_p[target])
        rho_f_p, _ = spearmanr(f_p[param], f_p[target])
        diffs[b] = rho_m_p - rho_f_p

    p = np.mean(np.abs(diffs) >= np.abs(obs_diff))
    return p


# we assume sex coded as 1.0 (male) and 0.0 (female)
sex_codes = [1.0, 0.0]
sex_labels = {1.0: "Male", 0.0: "Female"}

# sample sizes per (LVEF, sex)
n_by_lvef_sex = {}
for l_label, l_mask in lvef_groups.items():
    for s in sex_codes:
        mask = l_mask & (kept_filtered["sex"] == s)
        n_by_lvef_sex[(l_label, s)] = int(mask.sum())

# ===== collect rhos and p-values =====
row_data = []
all_abs_rhos = []

for pname, param in zip(pretty_names, param_order):
    rhos = {}  # (lvef_label, sex_code) -> rho
    pvals = {}  # lvef_label -> p (male vs female in that LVEF)
    for l_label, l_mask in lvef_groups.items():
        # correlations within each sex for this LVEF group
        for s in sex_codes:
            df_sub = kept_filtered.loc[l_mask & (kept_filtered["sex"] == s)]
            rho, _ = spearmanr(df_sub[param], df_sub["ensemble"], nan_policy="omit")
            rhos[(l_label, s)] = rho
            if not np.isnan(rho):
                all_abs_rhos.append(abs(rho))

        # p-value comparing male vs female in this LVEF group
        r_m = rhos[(l_label, 1.0)]
        r_f = rhos[(l_label, 0.0)]
        n_m = n_by_lvef_sex[(l_label, 1.0)]
        n_f = n_by_lvef_sex[(l_label, 0.0)]
        # p = pval_diff_spearman(r_m, n_m, r_f, n_f)
        # p = pval_diff_spearman(r_m, n_m, r_f, n_f)
        df_m = kept_filtered.loc[l_mask & (kept_filtered["sex"] == 1.0)]
        df_f = kept_filtered.loc[l_mask & (kept_filtered["sex"] == 0.0)]
        p = perm_test_diff_spearman(kept_filtered, l_mask, param, B=N_BOOTSTRAP)

        pvals[l_label] = p

    row_data.append(
        {
            "pname": pname,
            "rhos": rhos,
            "pvals": pvals,
        }
    )

max_abs_rho = max(all_abs_rhos) if all_abs_rhos else 0.0

# map |rho| to green!xx with:
#   xx = 50 for max |rho|
#   xx = 0 for |rho| <= 0.4 * max |rho|
#   linearly in between otherwise


def fmt_rho_colored(rho, max_abs):
    if pd.isna(rho) or max_abs == 0:
        return ""
    abs_r = abs(rho)
    half = 0.4 * max_abs
    if abs_r <= half:
        level = 0
    elif abs_r >= max_abs:
        level = 50
    else:
        level = int(round(50 * (abs_r - half) / (max_abs - half)))
    # wrap the number in \num
    return f"\\cellcolor{{green!{level}}} \\num{{{rho:.2f}}}"


# ===== build LaTeX rows =====
rows = []
for rd in row_data:
    cells = []
    for l_label in lvef_cols:
        # Male, Female rho cells
        for s in sex_codes:
            rho = rd["rhos"][(l_label, s)]
            cells.append(fmt_rho_colored(rho, max_abs_rho))

        # p-value cell for this LVEF group
        pval = rd["pvals"][l_label]
        if pd.isna(pval):
            p_str = ""
        else:
            if pval < 0.001:
                # only the numeric part is wrapped in \num
                p_str = r"<\num{0.001}"
            else:
                p_str = rf"\num{{{pval:.3f}}}"
        cells.append(p_str)

    row = "      " + " & ".join([rd["pname"]] + cells) + r" \\"
    rows.append(row)

# ----- table layout -----
# For each LVEF: Male, Female, p  -> 3 columns
n_lvef = len(lvef_cols)
cols_per_lvef = 3
total_data_cols = n_lvef * cols_per_lvef

col_spec_all = "l" + "r" * total_data_cols

# top header: LVEF subgroups
header_top = (
    "      & "
    + " & ".join(
        [f"\\multicolumn{{{cols_per_lvef}}}{{c}}{{{label}}}" for label in lvef_cols]
    )
    + r" \\"
)

# mid header: alternating sex and p under each LVEF
mid_cells = []
for _ in lvef_cols:
    mid_cells.extend(["Male", "Female", "p"])
header_mid = "      Parameter & " + " & ".join(mid_cells) + r" \\"

# cmidrules for each LVEF block
cmidrules = []
start = 2  # first data column
for i in range(n_lvef):
    end = start + cols_per_lvef - 1
    cmidrules.append(f"      \\cmidrule(lr){{{start}-{end}}}")
    start = end + 1
cmidrules_str = "\n".join(cmidrules)

combined_tabular = (
    f"   \\begin{{tabular}}{{{col_spec_all}}}\n"
    "      \\toprule\n"
    f"{header_top}\n"
    f"{cmidrules_str}\n"
    f"{header_mid}\n"
    "      \\midrule\n" + "\n".join(rows) + "\n"
    "      \\bottomrule\n"
    "   \\end{tabular}\n"
)

# avoid LaTeX issues with underscores in names
combined_tabular = combined_tabular.replace("_", " ")

latex_combined = (
    "\\begin{table*}\n"
    "   \\centering\n"
    "   \\begin{threeparttable}\n"
    "   \\caption{Spearman rank correlation between parameters and ensemble prediction by LVEF subgroup and sex.}\n"
    "   \\label{tab:spearman_corr_lvef_sex_6groups}\n"
    f"{combined_tabular}"
    "   \\begin{tablenotes}\\footnotesize\n"
    "   \\item Higher absolute $\\rho$ indicates stronger monotonic association. "
    "Within each LVEF subgroup, the $p$ columns report two-sided permutation test $p$-values for the difference in correlation between males and females.\n"
    "   \\end{tablenotes}\n"
    "   \\end{threeparttable}\n"
    "\\end{table*}\n"
)


with open("tables/spearman_table_lvef_sex_6groups.tex", "w") as f:
    f.write(latex_combined)

In [ ]:
# ================== Plotting (expects `columns` and `kept`) ==================
fig2, axs2 = plt.subplots(3, 5, figsize=(14, 8))
# plot_cols = ['age', 'LVEF', 'GLS', 'E/e'', 'MAPSE', 'LAv', 'TAPSE', "E"]

legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub = kept_filtered[kept_filtered[column].notna()]
    sub = sub.copy()
    x = sub[column].values
    y = sub["ensemble"].values  # probabilities in [0,1]
    y_diag = sub["Heart failure"].values  # ground-truth labels (0/1)

    ind = np.argsort(x)
    x = x[ind]
    y = y[ind]
    y_diag = y_diag[ind]

    # display limits (1st–99th pct)
    lims = np.percentile(x, [2, 98])
    lims = PLOT_LIMS.get(column, lims)
    ax.set_xlim(lims)

    qrs = 0.49
    xg_q, qlo, qhi = kernel_quantile_band_rankx(
        x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
    )
    qm = (qlo + qhi) / 2
    ax.plot(xg_q, qm, linewidth=2.0, color="C0", zorder=2, label="Median")
    qrs = 0.25
    xg_q, qlo, qhi = kernel_quantile_band_rankx(
        x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
    )
    ax.fill_between(
        xg_q,
        qlo,
        qhi,
        alpha=0.4,
        color="C0",
        zorder=1,
        label=f"{int(qrs*100)}--{int((1-qrs)*100)}% range",
        edgecolor="none",
    )

    ax.set_ylim(0, 1)
    unit = UNITS_MAPPING.get(column, "-")
    ax.set_title(f"{column.replace('_', ' ')}", fontsize=13)
    ax.set_xlabel(f"{unit}")
    if i in (0, 5, 10):
        # ax.set_ylabel("Model-predicted risk of HF (%)")
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])
    # ax.grid(axis='y')

    # # histogram axis behind the main one
    ax2 = ax.twinx()
    # make a kdeplot of density of x
    sns.kdeplot(
        x,
        ax=ax2,
        color="gray",
        fill=True,
        alpha=0.3,
        edgecolor="gray",
        linewidth=0.0,
        label="Parameter density",
        zorder=0,
        bw_adjust=0.5,
    )

    ax2.set_yticks([])
    ax2.patch.set_visible(False)  # remove white background
    ax.set_zorder(ax2.get_zorder() + 1)  # bring model curve to front
    ax.patch.set_visible(False)

    # remove spines of the histogram axis
    ax2.spines["top"].set_visible(False)
    ax2.spines["right"].set_visible(False)
    ax2.spines["bottom"].set_visible(False)
    ax2.spines["left"].set_visible(False)
    ax2.set_ylabel("")

    # ax as well (top and right)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        # add legend handles for ax2
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2


# single legend outside the grid
if legend_handles:
    # remove the legend border
    fig2.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=3,
        frameon=False,
        fontsize=12,
    )

# set y label for the entire figure
fig2.text(
    0.0,
    0.5,
    "Model-predicted risk of HF (%)",
    va="center",
    rotation="vertical",
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("figures/png/rankx_quartile_bands.png", dpi=300)
plt.savefig("figures/pgf/rankx_quartile_bands.pgf", bbox_inches="tight")
plt.show()

In [ ]:
fig2, axs2 = plt.subplots(3, 5, figsize=(15, 9))

# Define how to map gender values to labels and colors
sex_info = {
    0: {"label": "Female", "color": "C0"},
    1: {"label": "Male", "color": "C1"},
}

legend_handles, legend_labels = None, None

for i, (ax, column) in enumerate(zip(axs2.flatten(), plot_cols)):
    sub_all = kept_filtered[kept_filtered[column].notna()].copy()

    # display limits (1st–99th pct) based on the full sample
    x_all = sub_all[column].to_numpy()
    lims = np.percentile(x_all, [1, 99])
    ax.set_xlim(lims)
    ax.set_ylim(0, 1)
    ax.set_title(column.replace("_", " "))
    ax.set_xlabel("")

    if i in (0, 5, 10):
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["0", "25", "50", "75", "100"])
    else:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0], ["", "", "", "", ""])

    ax2 = ax.twinx()
    sns.kdeplot(
        x_all,
        ax=ax2,
        color="gray",
        fill=True,
        alpha=0.3,
        edgecolor="gray",
        linewidth=0.0,
        label="Parameter density",
        zorder=0,
        bw_adjust=0.5,
    )
    ax2.set_yticks([])
    ax2.patch.set_visible(False)
    ax.set_zorder(ax2.get_zorder() + 1)
    ax.patch.set_visible(False)

    # remove spines of the histogram axis
    for spine in ("top", "right", "bottom", "left"):
        ax2.spines[spine].set_visible(False)
    # remove top/right spines of main axis
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # --- Sex-specific curves ---
    for sex_val, info in sex_info.items():
        sub = sub_all[sub_all["sex"] == sex_val]
        if sub.empty:
            continue

        x = sub[column].to_numpy()
        y = sub["ensemble"].to_numpy()  # probabilities in [0,1]
        y_diag = sub[
            "Heart failure"
        ].to_numpy()  # ground-truth labels (0/1), unused here

        # sort by x
        ind = np.argsort(x)
        x = x[ind]
        y = y[ind]
        y_diag = y_diag[ind]

        # 98% central band around median
        qrs = 0.49
        xg_q, qlo, qhi = kernel_quantile_band_rankx(
            x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
        )
        qm = (qlo + qhi) / 2.0
        # median curve
        ax.plot(
            xg_q,
            qm,
            linewidth=2.0,
            color=info["color"],
            zorder=2,
            label=f"{info['label']} median",
        )

        # 50% central band
        qrs = 0.25
        xg_q, qlo, qhi = kernel_quantile_band_rankx(
            x, y, qs=(qrs, 1 - qrs), n_grid=N_GRID, const=1.0
        )
        ax.fill_between(
            xg_q,
            qlo,
            qhi,
            alpha=0.25,
            color=info["color"],
            zorder=1,
            edgecolor="none",
            label=f"{info['label']} {int(qrs*100)}–{int((1-qrs)*100)}% range",
        )

    # collect legend info only once
    if i == 0:
        legend_handles, legend_labels = ax.get_legend_handles_labels()
        handles2, labels2 = ax2.get_legend_handles_labels()
        legend_handles += handles2
        legend_labels += labels2

# single legend outside the grid
if legend_handles:
    fig2.legend(
        legend_handles,
        legend_labels,
        loc="upper center",
        ncol=5,
        frameon=False,
        fontsize=11,
    )

# set y label for the entire figure
fig2.text(
    0.0,
    0.5,
    "Model-predicted risk of HF (%)",
    va="center",
    rotation="vertical",
    fontsize=12,
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
# plt.savefig("figures/png/rankx_quartile_bands_sex_split.png", dpi=300)
# plt.savefig("figures/pgf/rankx_quartile_bands_sex_split.pgf", bbox_inches="tight")
plt.show()

In [ ]:
green_cmap = LinearSegmentedColormap.from_list(
    "green_diverging",
    ["#009900", "#FFFFFF", "#009900"],  # dark green → white → dark green
    N=256,
)

corr_spearman = pd.DataFrame(
    index=["ensemble"] + param_order,
    columns=["ensemble"] + param_order,
    data=np.nan,
)
for i, param1 in enumerate(["ensemble"] + param_order):
    for j, param2 in enumerate(["ensemble"] + param_order):
        # if j <= i:
        #     continue
        corr, _ = spearmanr(
            kept_filtered[param1], kept_filtered[param2], nan_policy="omit"
        )
        corr_spearman.at[param1, param2] = corr * 100  # in percent
        # corr_spearman.at[param2, param1] = corr * 100  # symmetric

plt.figure(figsize=(10, 10))
sns.heatmap(
    corr_spearman, annot=True, fmt=".0f", cmap=green_cmap, vmin=-80, vmax=80, cbar=False
)
plt.title("Spearman Rank Correlation (\%)")
plt.xticks(rotation=35, ha="right")

plt.yticks(rotation=0)
plt.yticks(
    ticks=np.array(list(range(len(corr_spearman.index)))) + 0.5,
    labels=[label.replace("_", " ") for label in corr_spearman.index],
)
plt.xticks(
    ticks=np.array(list(range(len(corr_spearman.columns)))) + 0.5,
    labels=[label.replace("_", " ") for label in corr_spearman.columns],
)
plt.savefig("figures/pgf/spearman_corr_matrix.pgf")
plt.show()

In [ ]:
def wrap_num_in_num(s: str):
    """Wrap numbers in \\num{} for siunitx; strip thousands separators inside \\num."""

    def repl(m):
        digits = m.group(1).replace(",", "")
        return f"\\num{{{digits}}}"

    return re.sub(r"(?<![A-Za-z0-9])([0-9][0-9,\.]*)", repl, s)


def fmt_n_pct(n, d):
    """Return 'n (p\\%)' string with escaped percent sign."""
    if d == 0 or pd.isna(d):
        return "0 (0\\%)"
    pct = 100.0 * n / d
    return f"{n:,} ({pct:.0f}\\%)"


def fmt_mean_sd(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    if a.empty:
        return ""
    return f"{a.mean():.1f} ({a.std(ddof=1):.1f})"


def fmt_median_iqr(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").dropna()
    if a.empty:
        return ""
    q1, q2, q3 = np.percentile(a, [25, 50, 75])
    return f"{q2:.0f} [{q1:.0f}, {q3:.0f}]"


def normalize_sex(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    if s in {"f", "female", "k"}:
        return "female"
    if s in {"m", "male"}:
        return "male"
    return np.nan


_REGEX_CACHE = {}


def any_match_regex(code_list, regex_list):
    if not isinstance(code_list, (list, tuple)):
        return False
    for pat in regex_list:
        if pat not in _REGEX_CACHE:
            _REGEX_CACHE[pat] = re.compile(pat)
        rx = _REGEX_CACHE[pat]
        for c in code_list:
            if isinstance(c, str) and rx.match(c):
                return True
    return False


def build_baseline_from_subdiag(
    subdiag: pd.DataFrame, n_total: int = None
) -> pd.DataFrame:
    expected = [
        "PERSONID",
        "OMSORGSEPISODEID",
        "KODE",
        "TEKST",
        "DIAGNOSETID",
        "HOVEDDIAGNOSE",
        "FOEDSELSNR",
        "time",
        "age",
        "sex",
        "IS_BASELINE",
    ]
    missing = set(expected) - set(subdiag.columns)
    if missing:
        raise ValueError(f"subdiag missing columns: {missing}")

    d = subdiag.copy()
    d["KODE"] = d["KODE"].astype(str).str.upper().str.strip()
    d["DIAGNOSETID"] = pd.to_datetime(d["DIAGNOSETID"], errors="coerce")
    d["time"] = pd.to_datetime(d["time"], errors="coerce")
    d["IS_BASELINE"] = (
        pd.to_numeric(d["IS_BASELINE"], errors="coerce").fillna(0).astype(int)
    )
    d = d.dropna(subset=["time"])

    # earliest echo per patient -> index row (for age/sex)
    idx_rows = (
        d.sort_values(["PERSONID", "time"])
        .drop_duplicates(subset=["PERSONID"], keep="first")
        .loc[:, ["PERSONID", "time", "age", "sex"]]
        .rename(columns={"time": "index_time"})
    )
    idx_rows["age"] = pd.to_numeric(idx_rows["age"], errors="coerce")
    idx_rows["Sex_norm"] = idx_rows["sex"].apply(normalize_sex)

    cohort_ids = idx_rows["PERSONID"].unique()
    N = n_total or len(cohort_ids)

    # collect codes at/before index (IS_BASELINE) and all-time
    codes_idx = (
        d.loc[d["IS_BASELINE"] == 1]
        .groupby("PERSONID")["KODE"]
        .apply(list)
        .reindex(cohort_ids, fill_value=[])
    )
    codes_ever = (
        d.groupby("PERSONID")["KODE"].apply(list).reindex(cohort_ids, fill_value=[])
    )

    # assemble rows
    rows = []
    rows.append(("Patients, n", "", f"{N:,}", ""))

    if idx_rows["age"].notna().any():
        rows.append(("Age", "", "", ""))
        rows.append(("\\quad Years, mean (SD)", "", fmt_mean_sd(idx_rows["age"]), ""))
        rows.append(
            ("\\quad Years, median [IQR]", "", fmt_median_iqr(idx_rows["age"]), "")
        )

    s = idx_rows["Sex_norm"]
    rows.append(("Sex", "", "", ""))
    rows.append(("\\quad Female", "", fmt_n_pct(int((s == "female").sum()), N), ""))
    rows.append(("\\quad Male", "", fmt_n_pct(int((s == "male").sum()), N), ""))
    miss = int(s.isna().sum())
    if miss > 0:
        rows.append(("\\quad Missing", "", fmt_n_pct(miss, N), ""))

    rows.append(("Comorbidities", "", "", ""))
    comorbid_sorted = []
    for name, regex_list in COMORBIDITY_REGEX.items():
        n_idx = int(codes_idx.apply(lambda cs: any_match_regex(cs, regex_list)).sum())
        n_evr = int(codes_ever.apply(lambda cs: any_match_regex(cs, regex_list)).sum())
        icd_label = ", ".join(COMORBIDITY_MAP.get(name, []))
        comorbid_sorted.append((name, icd_label, n_idx, n_evr))
    comorbid_sorted.sort(key=lambda x: x[2], reverse=True)
    for name, icd_label, n_idx, n_evr in comorbid_sorted:
        rows.append(
            (f"\\quad {name}", icd_label, fmt_n_pct(n_idx, N), fmt_n_pct(n_evr, N))
        )

    return pd.DataFrame(
        rows, columns=["Characteristic", "ICD-10", "On/before index", "All time"]
    )


# ---- build + LaTeX ----
baseline_df = build_baseline_from_subdiag(subdiag, n_total=len(kept_filtered))

caption = "Baseline characteristics for the full echocardiography cohort: age, sex, and comorbidities recorded before the index date and across all available time."
label = "tab:baseline_full_echo"

latex = dedent(
    f"""
\\begin{{table}}[!htbp]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\begin{{threeparttable}}
\\begin{{tabular}}{{@{{}}l l rr@{{}}}}
\\toprule
Characteristic & ICD-10 & At index & All time \\\\
\\midrule
"""
).lstrip()

for ch, icd, v1, v2 in baseline_df.itertuples(index=False):
    latex += f"{ch} & {wrap_num_in_num(icd)} & {wrap_num_in_num(v1)} & {wrap_num_in_num(v2)} \\\\\n"

latex += dedent(
    r"""
\bottomrule
\end{tabular}
\begin{tablenotes}[para]
\footnotesize
Before index includes diagnostic codes registered before the first ECG; All time includes diagnoses recorded after the index.
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
)

with open("tables/baseline_table_full_echo.tex", "w") as f:
    # drop empty rows from latex and indent all but first and last row
    new_latex = []
    for row in latex.splitlines():
        if row.strip() == "":
            continue
        if row.startswith("\\begin{table}") or row.startswith("\\end{table}"):
            new_latex.append(row)
        else:
            new_latex.append("\t" + row)
    new_latex[-1] = new_latex[-1].lstrip()
    latex = "\n".join(new_latex)
    f.write(latex)

In [ ]:
def _spearman_corr(x: np.ndarray, y: np.ndarray) -> float:
    if len(x) < 2 or len(y) < 2:
        return np.nan
    rho, _ = spearmanr(x, y)
    return abs(rho)


def plot_rankx_ci_by_sex(
    df: pd.DataFrame,
    plot_cols: Sequence[str] = ("LVEF", "PASP", "IVS", "PWT"),
    prob_col: str = "ensemble",
    sex_col: str = "sex",
    sexes: Sequence[str] = ("female", "male"),
    column_name_map: Optional[Dict[str, str]] = None,
    column_units_map: Optional[Dict[str, str]] = None,
    qs_inner: float = 0.975,
    qs_outer: float = 0.025,
    n_boot: int = 2000,
    random_state: Optional[int] = 0,
    min_n: int = 10,
    cache_dir: str = "./cache",
    cache_name: str = "rankx_ci_by_sex_ahus.csv",
) -> Tuple[plt.Figure, np.ndarray]:

    if column_name_map is None:
        column_name_map = {}
    if column_units_map is None:
        column_units_map = {}

    rng = np.random.default_rng(random_state)

    # NEW: container for saved statistics
    rows = []

    n_cols = len(plot_cols)
    fig, axs = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4), squeeze=False)
    axs = axs[0]

    x_positions = np.arange(len(sexes))
    legend_handles, legend_labels = None, None

    for ax_i, (ax, column) in enumerate(zip(axs, plot_cols)):
        meds = np.full(len(sexes), np.nan, dtype=float)
        los = np.full(len(sexes), np.nan, dtype=float)
        his = np.full(len(sexes), np.nan, dtype=float)
        ns = np.zeros(len(sexes), dtype=int)

        for s_idx, sex in enumerate(sexes):
            sub = (
                df.loc[df[sex_col] == sex, [column, prob_col]]
                .dropna()
                .reset_index(drop=True)
            )
            x = sub[column].to_numpy()
            y = sub[prob_col].to_numpy()
            n = len(x)
            ns[s_idx] = n

            if n < min_n:
                continue

            boot = np.empty(n_boot, dtype=float)
            for b in range(n_boot):
                idx = rng.integers(0, n, size=n)
                boot[b] = _spearman_corr(x[idx], y[idx])

            boot = boot[~np.isnan(boot)]
            if boot.size == 0:
                continue

            med = np.median(boot)  # change to np.mean(boot) if desired
            lo = np.quantile(boot, qs_outer)
            hi = np.quantile(boot, qs_inner)

            meds[s_idx] = med
            los[s_idx] = lo
            his[s_idx] = hi

            # NEW: store row
            sex_str = {0.0: "female", 1.0: "male"}[sex]
            rows.append(
                {
                    "variable": column,
                    "sex": sex_str,
                    "n": n,
                    "estimate": med,
                    "ci_low": lo,
                    "ci_high": hi,
                    "qs_outer": qs_outer,
                    "qs_inner": qs_inner,
                    "n_boot": n_boot,
                }
            )

        valid = ~np.isnan(meds)
        if np.any(valid):
            yerr = np.vstack([meds[valid] - los[valid], his[valid] - meds[valid]])
            ax.errorbar(
                x_positions[valid],
                meds[valid],
                yerr=yerr,
                fmt="o",
                capsize=4,
                label="Bootstrapped median ± CI",
            )

        ax.set_xticks(x_positions)
        ax.set_xticklabels(
            [f"{s}\n(n={ns[i]})" for i, s in enumerate(sexes)], fontsize=10
        )

        pretty = column_name_map.get(column, column)
        unit = column_units_map.get(column, "")
        ax.set_title(f"{pretty} [{unit}]" if unit else pretty, fontsize=11)

        if ax is axs[0]:
            ax.set_ylabel("Spearman |ρ|")
        else:
            ax.set_yticklabels([])
            ax.set_ylabel("")

        ax.set_ylim(0, 0.6)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        if ax_i == 0:
            legend_handles, legend_labels = ax.get_legend_handles_labels()

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            loc="upper center",
            ncol=2,
            frameon=False,
            fontsize=11,
        )

    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.92])

    # NEW: save results
    if rows:
        cache_path = Path(cache_dir)
        cache_path.mkdir(parents=True, exist_ok=True)
        out_df = pd.DataFrame(rows)
        out_df.to_csv(cache_path / cache_name, index=False)

    return fig, axs


df_plot = kept_filtered.copy()
fig, axs = plot_rankx_ci_by_sex(
    df_plot,
    plot_cols=("GLS", "LVEF", "PASP", "TRv", "IVS", "PWT"),
    prob_col="ensemble",
    sex_col="sex",
    sexes=(0.0, 1.0),
    column_units_map={
        "LVEF": "%",
        "PASP": "mmHg",
        "IVS": "cm",
        "PWT": "cm",
        "TRv": "m/s",
    },
    n_boot=N_BOOTSTRAP,
    random_state=0,
    min_n=25,
)

plt.show()